# LesionLens — Cloud GPU Training: TransAttUNet Baseline

**Project**: Efficient TransAttUnet for Skin Lesion Segmentation (ISIC 2016 Task 1)

This notebook configures, verifies, and runs Phase 2 baseline training of the original **TransAttUNet** (Chen et al., 2021) on cloud GPUs (e.g. Google Colab Tesla T4 / V100 / A100).

### Workflow Steps:
1. **GPU Verification**: Detect CUDA availability, GPU device name, compute capability, and VRAM.
2. **Repository & Dependencies**: Clone branch `bakia`, install dependencies, verify test suite.
3. **Dataset Acquisition & Preprocessing**: Download ISIC 2016 Part 1 archives, preprocess to 256x256, verify 720 train / 180 val splits.
4. **GPU Throughput & Smoke Test**: Run a 2-batch calibration pass to benchmark forward/backward throughput.
5. **Single-Epoch Calibration**: Execute 1 full epoch (720 train + 180 val) to obtain accurate timing and metrics.
6. **Inference Diagnostics**: Verify the checkpoint produces valid probability maps and binary masks.
7. **Persistent Export**: Save checkpoints and training logs to Google Drive or downloadable archive.
8. **Full Baseline Training**: Execute the remaining epochs with resumption enabled.

## 1. Verify CUDA & GPU Hardware Environment

In [ ]:
import os
import sys
import torch

print("=" * 60)
print("LESIONLENS — GPU ENVIRONMENT VERIFICATION")
print("=" * 60)

cuda_available = torch.cuda.is_available()
print(f"PyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")

if not cuda_available:
    raise SystemError(
        "CUDA is not available! Please enable GPU acceleration in Colab: "
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU / A100 GPU."
    )

device_count = torch.cuda.device_count()
current_device = torch.cuda.current_device()
device_name = torch.cuda.get_device_name(current_device)
capability = torch.cuda.get_device_capability(current_device)
total_mem_gb = torch.cuda.get_device_properties(current_device).total_memory / (1024 ** 3)

print(f"GPU Device Count:        {device_count}")
print(f"Primary GPU Name:        {device_name}")
print(f"Compute Capability:      {capability[0]}.{capability[1]}")
print(f"Total GPU Memory:        {total_mem_gb:.2f} GB")
print(f"cuDNN Enabled:           {torch.backends.cudnn.enabled}")
print(f"cuDNN Version:           {torch.backends.cudnn.version()}")
print("=" * 60)

## 2. Clone Repository and Install Dependencies

In [ ]:
# If running in Google Colab, clone the project repository
import os
from pathlib import Path

if not Path("models/transattunet.py").is_file():
    !git clone -b bakia https://github.com/aadhisesha/LesionLens.git
    %cd LesionLens
else:
    print("Already in LesionLens project directory:", os.getcwd())

# Install dependencies
!pip install -q -r requirements.txt

# Verify full test suite passes (36 tests)
!python -m pytest tests/ -v

## 3. Obtain ISIC 2016 Part 1 Dataset and Preprocess (256x256)

Downloads the 4 official archives from the ISIC Archive S3 server, extracts to `data/raw/`, resizes images (bilinear) and masks (nearest-neighbor) to 256x256, and generates reproducible 80/20 train/val splits (`data/splits/{train,val,test}.json`).

In [ ]:
# Run dataset preparation and validation pipeline
!python scripts/prepare_dataset.py --all

# Verify split manifest sizes
import json
from pathlib import Path

train_manifest = json.loads(Path("data/splits/train.json").read_text(encoding="utf-8"))
val_manifest = json.loads(Path("data/splits/val.json").read_text(encoding="utf-8"))
test_manifest = json.loads(Path("data/splits/test.json").read_text(encoding="utf-8"))

train_count = len(train_manifest.get("samples", train_manifest))
val_count = len(val_manifest.get("samples", val_manifest))
test_count = len(test_manifest.get("samples", test_manifest))

print("\n--- DATASET SPLIT VERIFICATION ---")
print(f"Training samples:    {train_count} (Expected: 720)")
print(f"Validation samples:  {val_count} (Expected: 180)")
print(f"Official test set:   {test_count} (Expected: 379 - QUARANTINED)")

assert train_count == 720, f"Expected 720 training samples, got {train_count}"
assert val_count == 180, f"Expected 180 validation samples, got {val_count}"
assert test_count == 379, f"Expected 379 test samples, got {test_count}"
print("Dataset splits verified successfully!")

## 4. Run GPU Throughput Benchmark & Smoke Test

Runs a short 2-batch smoke test on the GPU to measure:
- Exact forward/backward throughput (samples/sec and ms/batch)
- Peak GPU VRAM allocated
- Verification that weights update with finite loss
- Strict preservation of `checkpoints/smoke_test_checkpoint.pth`

In [ ]:
import time
import torch

# Reset peak memory tracking
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

# Execute 2-batch smoke test on CUDA
!python training/train.py --config configs/baseline.yaml --batch-size 4 --max-train-batches 2 --max-val-batches 2 --epochs 1 --device cuda

smoke_duration = time.time() - t0
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

print("\n--- GPU SMOKE TEST BENCHMARK ---")
print(f"Total execution time:    {smoke_duration:.2f} seconds")
print(f"Peak GPU Memory:         {peak_vram_mb:.1f} MB")
print(f"Checkpoint saved:        checkpoints/smoke_test_checkpoint.pth")
print("Smoke test verified on GPU!")

## 5. Single-Epoch Baseline Calibration (Full 720 Train + 180 Val Samples)

Trains for exactly **1 full epoch** on all 720 training samples (180 batches of size 4) and evaluates on all 180 validation samples (45 batches of size 4) to:
1. Measure the exact epoch duration on cloud GPU.
2. Estimate total duration for the remaining 29 epochs.
3. Save the baseline checkpoints:
   - `checkpoints/transattunet_baseline_best.pth`
   - `checkpoints/transattunet_baseline_latest.pth`
4. Record training loss, validation loss, Dice, and IoU.

In [ ]:
import time
import json
from pathlib import Path

print("Starting Epoch 1 baseline calibration on full splits...")
t_epoch_start = time.time()

!python training/train.py --config configs/baseline.yaml --batch-size 4 --epochs 1 --device cuda

epoch1_duration = time.time() - t_epoch_start

# Read recorded metrics
metrics_log = json.loads(Path("results/logs/train_metrics.json").read_text(encoding="utf-8"))
ep1 = metrics_log[0]

print("\n" + "=" * 60)
print("EPOCH 1 CALIBRATION RESULTS")
print("=" * 60)
print(f"Epoch 1 Duration:       {epoch1_duration:.1f}s ({epoch1_duration / 60:.2f} min)")
print(f"Training Loss:          {ep1['train_loss']:.4f}")
print(f"Validation Loss:        {ep1['val_loss']:.4f}")
print(f"Validation Dice:        {ep1['val_dice']:.4f}")
print(f"Validation IoU:         {ep1['val_iou']:.4f}")
print(f"Estimated 30-Epoch Run: {(epoch1_duration * 30) / 60:.1f} minutes total")
print("=" * 60)

## 6. Inference Diagnostics on Epoch 1 Checkpoint

Verifies that the checkpoint can be loaded and produces sensible predictions on a real validation image.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Select a sample image from the validation split
val_split = json.loads(Path("data/splits/val.json").read_text(encoding="utf-8"))
sample_image_path = val_split["samples"][0]["image_path"]
sample_mask_path = val_split["samples"][0]["mask_path"]

out_pred_path = "results/predictions/epoch1_val_prediction.png"

# Run inference pipeline
!python inference/predict.py \
    --checkpoint checkpoints/transattunet_baseline_best.pth \
    --image "$sample_image_path" \
    --output "$out_pred_path" \
    --threshold 0.5 \
    --device cuda

# Display visualization
if Path(out_pred_path).is_file():
    fig, ax = plt.subplots(figsize=(12, 4))
    img = Image.open(out_pred_path)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Epoch 1 Validation Prediction (Input | Prob Map | Binary Mask)")
    plt.show()

## 7. Export Checkpoints & Logs / Sync to Google Drive

Creates a compressed archive containing:
- `checkpoints/transattunet_baseline_best.pth`
- `checkpoints/transattunet_baseline_latest.pth`
- `results/logs/train_metrics.json`
- `results/logs/train_metrics.csv`
- `results/figures/training_curves.png`

In [ ]:
# Option A: Mount Google Drive for automatic persistence (uncomment if desired)
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/LesionLens_Checkpoints
# !cp -r checkpoints/* /content/drive/MyDrive/LesionLens_Checkpoints/
# !cp -r results/logs/* /content/drive/MyDrive/LesionLens_Checkpoints/

# Option B: Create a zip bundle for direct browser download
!zip -r lesionlens_baseline_phase2.zip checkpoints/ results/logs/ results/figures/ results/predictions/

try:
    from google.colab import files
    files.download("lesionlens_baseline_phase2.zip")
    print("Download initiated!")
except ImportError:
    print("Archive created at: lesionlens_baseline_phase2.zip")

## 8. Full Baseline Training (Remaining Epochs 2 -> 30)

> **STOP & REVIEW**: Only run this cell after reviewing the Epoch 1 metrics above and confirming approval to proceed.

This command resumes training seamlessly from `checkpoints/transattunet_baseline_latest.pth` at Epoch 2, carrying forward all metrics history and learning rate schedules.

In [ ]:
# Execute full 30-epoch training (resumes from latest checkpoint at Epoch 2)
!python training/train.py \
    --config configs/baseline.yaml \
    --batch-size 4 \
    --epochs 30 \
    --resume checkpoints/transattunet_baseline_latest.pth \
    --device cuda

# Re-export final checkpoints and training curves
!zip -r lesionlens_baseline_final_30ep.zip checkpoints/ results/
try:
    from google.colab import files
    files.download("lesionlens_baseline_final_30ep.zip")
except ImportError:
    pass